# 01 — Common Crawl Index Query

Query the Common Crawl index to enumerate .ie domains and discover which pages are available.
This notebook has three approaches in increasing complexity:

- **Option A (start here)**: CC Index Server API — free, no AWS needed, limited to ~50K records per query
- **Option B**: DuckDB local query — downloads parquet index files, queries offline (~300GB, free)
- **Option C**: AWS Athena — scalable SQL over the full index, ~$1.50/query

**Output**: `data/raw/ie_domains.jsonl` — one CC index record per line for .ie pages

In [1]:
import sys
sys.path.insert(0, '..')

import json
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('../.env', override=True)

DATA_DIR = Path('../data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Environment ready')

Environment ready


## Option A: CC Index Server API (Recommended Start)

No AWS account needed. Good for validating coverage and getting a representative sample.
Rate-limit: be polite — add sleep between paginated requests.

Available crawls: https://index.commoncrawl.org/collinfo.json

In [2]:
from src.common_crawl import query_cc_index, get_ie_domains, deduplicate_by_domain

CRAWL = 'CC-MAIN-2026-08'  # Update to most recent crawl

# Sanity check: how many .ie pages are in the index?
# Start with a small limit to verify connectivity
sample = query_cc_index('.ie', crawl=CRAWL, limit=100)
print(f'Sample query returned {len(sample)} records')
if sample:
    print('Sample record:', json.dumps(sample[0], indent=2))

Sample query returned 100 records
Sample record: {
  "urlkey": "ie,00)/",
  "timestamp": "20260218031107",
  "url": "http://00.ie/",
  "mime": "text/html",
  "mime-detected": "application/xhtml+xml",
  "status": "200",
  "digest": "U6KVBASBWKTWN3J6YLSFSW7GHEWMCU7Y",
  "length": "2015",
  "offset": "890",
  "filename": "crawl-data/CC-MAIN-2026-08/segments/1770395864156.75/warc/CC-MAIN-20260218015747-20260218045747-00158.warc.gz",
  "languages": "eng",
  "encoding": "UTF-8"
}


In [3]:
# Full query — up to 50K records via pagination
# For the full Irish web, use Option B or C below
ie_records = get_ie_domains(crawl=CRAWL, limit=50_000)
print(f'Total .ie records: {len(ie_records)}')

# Deduplicate to one record per domain
by_domain = deduplicate_by_domain(ie_records)
print(f'Unique domains: {len(by_domain)}')

Total .ie records: 5000
Unique domains: 118


In [4]:
# Save to JSONL
out_path = DATA_DIR / 'ie_domains_cc_api.jsonl'
with open(out_path, 'w') as f:
    for domain, record in by_domain.items():
        record['_domain'] = domain
        f.write(json.dumps(record) + '\n')

print(f'Saved {len(by_domain)} domain records to {out_path}')

Saved 118 domain records to ../data/raw/ie_domains_cc_api.jsonl


## Option B: DuckDB Local Query

Download CC columnar index and query offline. Zero cost but requires ~300GB disk.
Use the `httpfs` extension to query S3 directly without downloading everything.

This approach scales to the full index and is reproducible.

In [5]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# Common Crawl S3 is a public bucket — no AWS credentials needed.
# Configure DuckDB for anonymous S3 access.
con.execute("""
    SET s3_region = 'us-east-1';
    SET s3_access_key_id = '';
    SET s3_secret_access_key = '';
    SET s3_url_style = 'path';
""")

CRAWL_ID = 'CC-MAIN-2026-08'

query = f"""
SELECT
    url_host_registered_domain AS domain,
    COUNT(*) AS page_count,
    MIN(url) AS sample_url,
    MIN(warc_filename) AS warc_filename,
    MIN(CAST(warc_record_offset AS BIGINT)) AS warc_record_offset,
    MIN(CAST(warc_record_length AS BIGINT)) AS warc_record_length
FROM read_parquet(
    's3://commoncrawl/cc-index/table/cc-main/warc/crawl={CRAWL_ID}/subset=warc/*.parquet',
    hive_partitioning=1
)
WHERE url_host_tld = 'ie'
  AND fetch_status = 200
GROUP BY url_host_registered_domain
ORDER BY page_count DESC
LIMIT 200000
"""

# To run: uncomment the three lines below.
# Takes a few minutes; free from anywhere, fastest from EC2 us-east-1.
# result = con.execute(query).fetchdf()
# print(f"Found {len(result)} .ie domains")
# result.head(20)

print("DuckDB configured for anonymous CC S3 access. Uncomment the execute() lines to run.")


DuckDB configured for anonymous CC S3 access. Uncomment the execute() lines to run.


## Option C: AWS Athena

Most scalable option. ~$1.50 per query. Requires AWS account and Athena setup.
Full CC index setup: https://commoncrawl.org/blog/index-to-warc-files-and-urls-in-columnar-format

In [ ]:
# AWS Athena query (requires boto3 + configured AWS credentials)
# pip install boto3

ATHENA_QUERY = """
SELECT url_host_registered_domain,
       COUNT(*) as page_count,
       MIN(url) as sample_url,
       MIN(warc_filename) as warc_filename,
       MIN(warc_record_offset) as warc_record_offset,
       MIN(warc_record_length) as warc_record_length
FROM "ccindex"."ccindex"
WHERE crawl = 'CC-MAIN-2026-08'
  AND subset = 'warc'
  AND url_host_tld = 'ie'
  AND fetch_status = 200
GROUP BY url_host_registered_domain
ORDER BY page_count DESC
"""

def run_athena_query(query: str, output_location: str, database: str = 'ccindex'):
    import boto3
    client = boto3.client('athena', region_name=os.getenv('AWS_DEFAULT_REGION', 'us-east-1'))

    response = client.start_query_execution(
        QueryString=query,
        QueryExecutionContext={'Database': database},
        ResultConfiguration={'OutputLocation': output_location},
    )
    execution_id = response['QueryExecutionId']
    print(f'Query started: {execution_id}')

    import time
    while True:
        status = client.get_query_execution(QueryExecutionId=execution_id)
        state = status['QueryExecution']['Status']['State']
        if state in ('SUCCEEDED', 'FAILED', 'CANCELLED'):
            print(f'Query {state}')
            break
        time.sleep(5)

    if state == 'SUCCEEDED':
        paginator = client.get_paginator('get_query_results')
        results = []
        for page in paginator.paginate(QueryExecutionId=execution_id):
            results.extend(page['ResultSet']['Rows'])
        return results

print('Athena function defined. Call run_athena_query(ATHENA_QUERY, "s3://your-bucket/results/") to execute.')

## Analysis: Domain Coverage

In [ ]:
# Load whichever dataset you produced above and analyse
import pandas as pd

records = []
with open(DATA_DIR / 'ie_domains_cc_api.jsonl') as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(f'Total unique .ie domains in CC: {len(df)}')
print(df.head(10))

In [ ]:
import matplotlib.pyplot as plt

# Sanity check: status code distribution
if 'status' in df.columns:
    df['status'].value_counts().plot(kind='bar', title='HTTP Status Codes', figsize=(8, 4))
    plt.tight_layout()
    plt.show()

print(f'Domains ready for WARC extraction: {len(df)}')
print('\nNext step: 02_wdc_schema_download.ipynb')